# Detect doublets using Scrublet

Our cell hashing processes catch and remove many doublets that are generated by mixing of cells from different samples, but some percentage of doublets (~7-8%) will be generated by collisions of cells from the same sample, and will not be detected.

To detect and remove these, we'll utilize the Scrublet package. Scrublet's process for doublet identification is described in this publication:

Wolock, S. L., Lopez, R. & Klein, A. M. Scrublet: Computational Identification of Cell Doublets in Single-Cell Transcriptomic Data. Cell Syst 8, 281–291.e9 (2019)

We'll use scrublet's integration into the scanpy package's [scanpy.external tools](https://scanpy.readthedocs.io/en/stable/generated/scanpy.external.pp.scrublet.html#scanpy.external.pp.scrublet).

Here, we apply scrublet to each of our .h5 files, and store the results in HISE for downstream analysis steps.

## Load Packages

`anndata`: Data structures for scRNA-seq  
`concurrent.futures`: parallelization methods  
`datetime`: date and time functions  
`h5py`: HDF5 file I/O  
`hisepy`: The HISE SDK for Python  
`numpy`: Mathematical data structures and computation  
`os`: operating system calls  
`pandas`: DataFrame data structures  
`re`: Regular expressions  
`scanpy`: scRNA-seq analysis  
`scipy.sparse`: Spare matrix data structures  

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import anndata
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import h5py
import hisepy
import numpy as np
import os
import pandas as pd 
import re
import scanpy as sc
import scanpy.external as sce
import scipy.sparse as scs

## Read sample metadata from HISE

In [2]:
sample_meta_file_uuid = 'cb01d5b1-40d8-4940-b61c-86d78de4528b'
file_query = hisepy.reader.read_files(
    [sample_meta_file_uuid]
)

In [3]:
meta_data = file_query['values']

In [4]:
meta_data.shape

(218, 33)

## Helper functions

These functions will retrieve data from HISE and read as AnnData for use with scrublet, and for reading and applying scrublet to each file.

In [5]:
# define a function to read count data
def read_mat(h5_con):
    mat = scs.csc_matrix(
        (h5_con['matrix']['data'][:], # Count values
         h5_con['matrix']['indices'][:], # Row indices
         h5_con['matrix']['indptr'][:]), # Pointers for column positions
        shape = tuple(h5_con['matrix']['shape'][:]) # Matrix dimensions
    )
    return mat

# define a function to read obeservation metadata (i.e. cell metadata)
def read_obs(h5con):
    bc = h5con['matrix']['barcodes'][:]
    bc = [x.decode('UTF-8') for x in bc]

    # Initialized the DataFrame with cell barcodes
    obs_df = pd.DataFrame({ 'barcodes' : bc })
    obs_df = obs_df.set_index('barcodes', drop = False)
    obs_df['barcodes'] = obs_df['barcodes'].astype("category")

    return obs_df

# define a function to construct anndata object from a h5 file
def read_h5_anndata(h5_con):
    #h5_con = h5py.File(h5_file, mode = 'r')
    # extract the expression matrix
    mat = read_mat(h5_con)
    # extract gene names
    genes = h5_con['matrix']['features']['name'][:]
    genes = [x.decode('UTF-8') for x in genes]
    # extract metadata
    obs_df = read_obs(h5_con)
    # construct anndata
    adata = anndata.AnnData(mat.T,
                            obs = obs_df)
    # make sure the gene names aligned
    adata.var_names = genes

    adata.var_names_make_unique()
    return adata

In [6]:
def get_adata(uuid):
    # Load the file using HISE
    res = hisepy.reader.read_files([uuid])

    # If there's an error, read_files returns a list instead of a dictionary.
    # We should raise and exception with the message when this happens.
    if(isinstance(res, list)):
        error_message = res[0]['message']
        raise Exception(error_message)
    
    # Read the file to adata
    h5_con = res['values'][0]
    adata = read_h5_anndata(h5_con)
    
    # Clean up the file now that we're done with it
    h5_con.close()

    return(adata)

In [7]:
def process_file(file_uuid):
    adata = get_adata(file_uuid)
    sc.external.pp.scrublet(
        adata,
        random_state = 3030,
        verbose = False
    )
    result = adata.obs[['barcodes','predicted_doublet','doublet_score']]
    return result

In [8]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Apply to each sample in parallel

Here, we'll use `concurrent.futures` to apply the function above to our samples in parallel.

In [9]:
results = []
file_uuids = meta_data['file.id'].tolist()
with ThreadPoolExecutor(max_workers = 20) as executor:  
    for result in executor.map(process_file, file_uuids):
        results.append(result)

Assemble results from all files

In [10]:
len(file_uuids)

218

In [11]:
final_result = pd.concat(results, ignore_index = True)

In [12]:
final_result

,barcodes,predicted_doublet,doublet_score
0,b9993ec4872b11ebb599de5a8c2059ae,False,0.060170
1,b99942fc872b11ebb599de5a8c2059ae,False,0.013071
2,b9996f66872b11ebb599de5a8c2059ae,False,0.120700
3,b99971c8872b11ebb599de5a8c2059ae,False,0.018831
4,b9997290872b11ebb599de5a8c2059ae,False,0.026079
...,...,...,...
3910923,6ef6f4809ab411ed964b0aeb18fc086d,False,0.012573
3910924,6ef6f5ca9ab411ed964b0aeb18fc086d,False,0.260341
3910925,6ef706789ab411ed964b0aeb18fc086d,False,0.090728
3910926,6ef70af69ab411ed964b0aeb18fc086d,False,0.063895


What's the total doublet count and fraction of cells?

In [13]:
prediction_counts = final_result['predicted_doublet'].value_counts()
prediction_counts

predicted_doublet
False    3889096
True       21832
Name: count, dtype: int64

In [14]:
prediction_counts[1] / sum(prediction_counts)

0.005582306808000557

## Save results to files

In [15]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [16]:
out_csv = 'output/up1_scrublet_results_{d}.csv'.format(d = date.today())
final_result.to_csv(out_csv)

In [17]:
final_result.shape

(3910928, 3)

In [18]:
final_result.head()

,barcodes,predicted_doublet,doublet_score
0,b9993ec4872b11ebb599de5a8c2059ae,False,0.060170
1,b99942fc872b11ebb599de5a8c2059ae,False,0.013071
2,b9996f66872b11ebb599de5a8c2059ae,False,0.120700
3,b99971c8872b11ebb599de5a8c2059ae,False,0.018831
4,b9997290872b11ebb599de5a8c2059ae,False,0.026079


In [19]:
out_parquet = 'output/up1_scrublet_results_{d}.parquet'.format(d = date.today())
final_result.to_parquet(out_parquet)

## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [20]:
ss = hisepy.get_study_spaces()

In [24]:

print(ss[0]['name'])
print(ss[0]['id'])
study_space_uuid = ss[0]['id']

title = '02 Scrublet results {d}'.format(d = date.today())

UP1 scRNAseq Study
0b6bf907-6985-40e0-944d-677ac932677f


In [25]:
search_id = element_id()
search_id

'magnesium-osmium-arsenic'

In [26]:
in_files = [sample_meta_file_uuid] + meta_data['file.id'].to_list()

In [27]:
len(in_files)

219

In [28]:
out_files = [out_csv, out_parquet]
out_files

['output/up1_scrublet_results_2024-08-16.csv',
 'output/up1_scrublet_results_2024-08-16.parquet']

In [29]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id,
    do_prompt = False
)

{'trace_id': '7f464887-e515-49a9-a5d2-c82e036915d9',
 'files': ['output/up1_scrublet_results_2024-08-16.csv',
  'output/up1_scrublet_results_2024-08-16.parquet']}

In [ ]:
import session_info
session_info.show()